# Genixcure PacBio HiFi — EcoRV CIGAR 독립 검증 v3

**목표:** 기존 12,000 unique 파일럿의 실제 CIGAR를 참조서열 및 FASTA와 대조하고, ITR 96bp, 두 개의 12bp 가변 구간, 신규 21bp 삽입을 별도 검증. 원본 데이터/기존 파일럿은 변경하지 않습니다.

**주의:** 가중 unique 표본은 전체 HiFi read population의 무작위 표본이 아닙니다.

In [ ]:
from google.colab import drive
from pathlib import Path
import json,subprocess,pandas as pd
drive.mount("/content/drive")
BASE=Path("/content/drive/MyDrive/Genixcure_PCR_Merge_Analysis/PacBio_PLA_6_13/Reference_EcoRV_Linear")
PILOT=BASE/"Alignment_Pilot_v1"
V2=BASE/"Alignment_Pilot_v2_CIGAR_Audit"
OUT=BASE/"Alignment_Pilot_v3_Independent_Audit"
SCRIPT=OUT/"Genixcure_PacBio_EcoRV_Independent_CIGAR_Audit_v3.py"
REF=BASE/"plasmid_EcoRV_linear_minimap2.fasta"
FASTA=PILOT/"weighted_pilot_reads.fasta"
META=PILOT/"sample_metadata.csv"
PAF=V2/"pilot_EcoRV_linear_eqx_cigar.paf"
for p in (SCRIPT,REF,FASTA,META,PAF):
    assert p.exists(),f"Input missing: {p}"
print("Verified reference and 12k pilot inputs")

In [ ]:
subprocess.run(["python",str(SCRIPT),"--self-test"],check=True)
cmd=["python",str(SCRIPT),"--reference",str(REF),"--fasta",str(FASTA),"--paf",str(PAF),"--metadata",str(META),"--out",str(OUT)]
subprocess.run(cmd,check=True)
print("Independent v3 CIGAR audit completed")

In [ ]:
d=json.loads((OUT/"independent_CIGAR_audit_v3.json").read_text())
assert d["sample_unique"]==12000
assert d["mapped_unique"]==11946
assert d["counts"]["itr96"]==10071
assert d["counts"]["extra21"]==5161
print("Aligned substitution identity (excludes indels):",round(100*d["aligned_substitution_identity_excluding_indels"],4),"%")
display(pd.DataFrame([{"event":k,"unique":v,"pilot_%":round(v/12000*100,2)} for k,v in d["counts"].items()]))
display(pd.DataFrame(d["events_top_insertions"][:10]))
print("All verified. Output:",OUT)